In [1]:
import sys
print([p for p in sys.path if 'spark' in p])

['/opt/spark/python', '/opt/spark/python/lib/py4j-0.10.9.9-src.zip']


In [2]:
import os
from pyspark.sql import SparkSession

SPARK_MASTER = os.environ.get('SPARK_MASTER', 'spark://spark-master:7077')

spark = SparkSession.builder \
    .appName('Demo Spark - Arquitecto de Datos') \
    .master(SPARK_MASTER) \
    .config('spark.executor.memory', '512m') \
    .config('spark.driver.host', 'jupyter') \
    .getOrCreate()

print(f'Spark version : {spark.version}')
print(f'Master        : {spark.sparkContext.master}')
print(f'App ID        : {spark.sparkContext.applicationId}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 16:03:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 4.1.1
Master        : spark://spark-master:7077
App ID        : app-20260429160345-0000


In [3]:
texto = [
    'hadoop spark kafka flink',
    'spark es rapido spark es potente',
    'kafka es distribuido kafka es escalable',
    'hadoop hdfs yarn mapreduce',
    'spark dataframe spark rdd spark streaming'
]

word_count = (
    spark.sparkContext.parallelize(texto)
    .flatMap(lambda line: line.split())
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

print('Top palabras:')
for word, count in word_count.take(10):
    print(f'  {word:20s} {count}')

Top palabras:
  spark                6
  es                   4
  kafka                3
  hadoop               2
  rapido               1
  hdfs                 1
  yarn                 1
  streaming            1
  flink                1
  potente              1


In [4]:
from pyspark.sql import Row
from pyspark.sql.functions import avg, count, round as spark_round, col

datos = [
    Row(region='Norte', producto='A', importe=1200.0, unidades=10),
    Row(region='Norte', producto='B', importe=800.0,  unidades=5),
    Row(region='Sur',   producto='A', importe=950.0,  unidades=8),
    Row(region='Sur',   producto='C', importe=1500.0, unidades=12),
    Row(region='Este',  producto='B', importe=600.0,  unidades=4),
    Row(region='Este',  producto='A', importe=1100.0, unidades=9),
    Row(region='Oeste', producto='C', importe=2000.0, unidades=15),
    Row(region='Oeste', producto='A', importe=750.0,  unidades=6),
]

df = spark.createDataFrame(datos)
df.printSchema()
df.show()

root
 |-- region: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- importe: double (nullable = true)
 |-- unidades: long (nullable = true)

+------+--------+-------+--------+
|region|producto|importe|unidades|
+------+--------+-------+--------+
| Norte|       A| 1200.0|      10|
| Norte|       B|  800.0|       5|
|   Sur|       A|  950.0|       8|
|   Sur|       C| 1500.0|      12|
|  Este|       B|  600.0|       4|
|  Este|       A| 1100.0|       9|
| Oeste|       C| 2000.0|      15|
| Oeste|       A|  750.0|       6|
+------+--------+-------+--------+



In [5]:
resumen = df.groupBy('region').agg(
    spark_round(avg('importe'), 2).alias('importe_medio'),
    count('*').alias('num_ventas'),
    spark_round(avg('unidades'), 1).alias('unidades_media')
).orderBy(col('importe_medio').desc())

resumen.show()

+------+-------------+----------+--------------+
|region|importe_medio|num_ventas|unidades_media|
+------+-------------+----------+--------------+
| Oeste|       1375.0|         2|          10.5|
|   Sur|       1225.0|         2|          10.0|
| Norte|       1000.0|         2|           7.5|
|  Este|        850.0|         2|           6.5|
+------+-------------+----------+--------------+



In [6]:
df.createOrReplaceTempView('ventas')

spark.sql("""
    SELECT producto,
           SUM(importe)  AS total_importe,
           SUM(unidades) AS total_unidades,
           COUNT(*)      AS num_regiones
    FROM ventas
    GROUP BY producto
    ORDER BY total_importe DESC
""").show()

+--------+-------------+--------------+------------+
|producto|total_importe|total_unidades|num_regiones|
+--------+-------------+--------------+------------+
|       A|       4000.0|            33|           4|
|       C|       3500.0|            27|           2|
|       B|       1400.0|             9|           2|
+--------+-------------+--------------+------------+



In [7]:
spark.sql("SELECT region, SUM(importe) FROM ventas GROUP BY region").explain(mode='formatted')

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [4]: [region#0, producto#1, importe#2, unidades#3L]
Arguments: [region#0, producto#1, importe#2, unidades#3L], MapPartitionsRDD[17] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [2]: [region#0, importe#2]
Input [4]: [region#0, producto#1, importe#2, unidades#3L]

(3) HashAggregate
Input [2]: [region#0, importe#2]
Keys [1]: [region#0]
Functions [1]: [partial_sum(importe#2)]
Aggregate Attributes [1]: [sum#83]
Results [2]: [region#0, sum#84]

(4) Exchange
Input [2]: [region#0, sum#84]
Arguments: hashpartitioning(region#0, 200), ENSURE_REQUIREMENTS, [plan_id=140]

(5) HashAggregate
Input [2]: [region#0, sum#84]
Keys [1]: [region#0]
Functions [1]: [sum(importe#2)]
Aggregate Attributes [1]: [sum(importe#2)#81]
Results [2]: [r

In [8]:
spark.stop()
print('Sesión Spark cerrada.')

Sesión Spark cerrada.
